In [22]:
import glob
import os

import numpy as np
import pandas as pd

from agent_k.config.logger import logger
from agent_k.config.schemas import InferlinkEvalColumns

INFERLINK_43_101_ANNOTATIONS_DIR = "data/raw/ground_truth/inferlink_43-101_annotations"

In [23]:
"""
Read all metadata.csv files in the specified directory. Enrich the dataframe with the cdr_record_id.
"""

# Find all metadata.csv files recursively
metadata_files = sorted(
    glob.glob(f"{INFERLINK_43_101_ANNOTATIONS_DIR}/**/metadata.csv", recursive=True)
)

logger.info(f"Found {len(metadata_files)} metadata.csv files")

master_metadata_df = pd.DataFrame()

# Process each file
for file_path in metadata_files:
    try:
        # Get the parent directory name (i.e. the cdr_record_id)
        parent_dir = os.path.basename(os.path.dirname(file_path))

        df_metadata = pd.read_csv(file_path)
        df_metadata.insert(0, InferlinkEvalColumns.CDR_RECORD_ID.value, parent_dir)
        master_metadata_df = pd.concat(
            [master_metadata_df, df_metadata], ignore_index=True
        )

    except Exception as e:
        logger.error(f"Error processing {file_path}: {e}")

logger.info("Processing complete!")
master_metadata_df.head()

2025-06-27 22:39:54.552 | INFO     | __main__:<module>:8 - Found 86 metadata.csv files


2025-06-27 22:39:54.643 | INFO     | __main__:<module>:25 - Processing complete!


,cdr_record_id,country_observed_name,state_or_province_observed_name,mining_name,authors,year,month
0,021639dd82455469880eabc73dc0bd32cac7bd5b0483e2...,Slovakia,Bratislava Region,Podpolom Property,Stephen Kenwood,2006,3
1,02163e7674b5c838a0efd7d2e80116e36ba01cbdb1d892...,Canada,British Columbia,Chu Chua Property,"Michael Dufresne, Kris Raffle, Steve Nicholls",2013,9
2,021ee3d3ec6928901d9427796372edaada0d332f15d030...,Canada,British Columbia,Red Chris Property,"Greg Gillstrom, Steve Robertson",2010,5
3,023b6fbcbcdc4222c843a68f1125f8a321d7e183e40297...,Canada,British Columbia,Ball Creek Property,Darcy E.L. Baker,2012,6
4,023cfb1faaecf8d1d8db5a7c3449bd24c7f4a6fc994839...,United States,Alaska,Gold Hill Project,Robert S. Friberg,2004,6


In [24]:
"""
Read all mineral_inventory_minimal.csv files in the specified directory. Enrich the dataframe with the cdr_record_id.
"""
# Find all mineral_inventory_minimal.csv files recursively
inventory_files = sorted(
    glob.glob(
        f"{INFERLINK_43_101_ANNOTATIONS_DIR}/**/mineral_inventory_minimal.csv",
        recursive=True,
    )
)

logger.info(f"Found {len(inventory_files)} mineral_inventory_minimal.csv files")

master_inventory_df = pd.DataFrame()

# Process each file
for file_path in inventory_files:
    try:
        # Get the parent directory name (which will be the cdr_record_id)
        parent_dir = os.path.basename(os.path.dirname(file_path))
        parent_parent_dir = os.path.basename(
            os.path.dirname(os.path.dirname(file_path))
        )

        # Read the CSV file
        df_inventory = pd.read_csv(file_path)

        # Insert the new column at the beginning
        df_inventory.insert(0, InferlinkEvalColumns.CDR_RECORD_ID.value, parent_dir)
        df_inventory.insert(
            1, InferlinkEvalColumns.MAIN_COMMODITY.value, parent_parent_dir
        )

        # If the df is empty, append a placeholder row with cdr_record_id
        if df_inventory.empty:
            df_inventory = pd.DataFrame(
                {
                    InferlinkEvalColumns.CDR_RECORD_ID.value: [parent_dir],
                    InferlinkEvalColumns.MAIN_COMMODITY.value: [parent_parent_dir],
                }
            )

        # Append to master inventory dataframe
        master_inventory_df = pd.concat(
            [master_inventory_df, df_inventory], ignore_index=True
        )

    except Exception as e:
        logger.error(f"Error processing {file_path}: {e}")

master_inventory_df[InferlinkEvalColumns.COMMODITY_OBSERVED_NAME.value] = (
    master_inventory_df[InferlinkEvalColumns.COMMODITY_OBSERVED_NAME.value]
    .str.lower()
    .str.strip()
)

print("Processing complete!")
master_inventory_df.head()

2025-06-27 22:39:54.659 | INFO     | __main__:<module>:9 - Found 86 mineral_inventory_minimal.csv files


Processing complete!


,cdr_record_id,main_commodity,commodity_observed_name,category_observed_name,ore_unit_observed_name,ore_value,grade_unit_observed_name,grade_value,cutoff_grade_unit_observed_name,cutoff_grade_value,contained_metal,zone
0,021639dd82455469880eabc73dc0bd32cac7bd5b0483e2...,copper,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,02163e7674b5c838a0efd7d2e80116e36ba01cbdb1d892...,copper,copper,indicated,tonnes,2827047.0,percent,1.90,grams per tonne,0.0,NaN,Chu Chua
2,02163e7674b5c838a0efd7d2e80116e36ba01cbdb1d892...,copper,zinc,indicated,tonnes,2827047.0,percent,0.32,grams per tonne,0.0,NaN,Chu Chua
3,02163e7674b5c838a0efd7d2e80116e36ba01cbdb1d892...,copper,silver,indicated,tonnes,2827047.0,grams per tonne,8.96,grams per tonne,0.0,NaN,Chu Chua
4,02163e7674b5c838a0efd7d2e80116e36ba01cbdb1d892...,copper,gold,indicated,tonnes,2827047.0,grams per tonne,0.47,grams per tonne,0.0,NaN,Chu Chua


In [25]:
def normalize_ore_units(df):
    """
    Normalize ore units to million tonnes. Creates two new columns:
    - normalized_ore_value: The ore value converted to million tonnes
    - normalized_ore_unit: Set to 'Mt' (million tonnes)

    Args:
        df (pd.DataFrame): DataFrame containing ore values and units

    Returns:
        pd.DataFrame: DataFrame with added normalization columns
    """
    # Create copies of the original columns
    df["normalized_ore_value"] = df["ore_value"].copy()
    df["normalized_ore_unit"] = df["ore_unit_observed_name"].copy()

    # Define conversion factors to million tonnes (Mt)
    conversion_factors = {
        "mt": 1.0,  # Million tonnes
        "million tonnes": 1.0,  # Million tonnes (spelled out)
        "gt": 1000.0,  # Billion tonnes (Giga tonnes)
        "kt": 0.001,  # Thousand tonnes (kilo tonnes)
        "thousand tonnes": 0.001,  # Thousand tonnes (spelled out)
        "t": 0.000001,  # Tonnes
        "tonnes": 0.000001,  # Tonnes (spelled out)
        "tonnage": 0.000001,  # Tonnage (assuming equivalent to tonnes)
        "metric tons": 0.000001,  # Metric tons
        "tons": 0.0000009072,  # Tons (assuming short tons)
        "ton": 0.0000009072,  # Ton (assuming short tons)
        "short tons": 0.0000009072,  # Short tons (1 short ton = 0.9072 metric tonnes)
    }

    # Apply conversion
    for idx, row in df.iterrows():
        try:
            if pd.notna(row["ore_unit_observed_name"]) and pd.notna(row["ore_value"]):
                unit = (
                    row["ore_unit_observed_name"].strip().lower()
                )  # Convert to lowercase for case-insensitive matching
                if unit in conversion_factors:
                    # Convert the value to million tonnes
                    df.at[idx, "normalized_ore_value"] = (
                        float(row["ore_value"]) * conversion_factors[unit]
                    )
                    df.at[idx, "normalized_ore_unit"] = "Mt"
                else:
                    # If unit not recognized raise an error
                    raise ValueError(
                        f"Unit '{row['ore_unit_observed_name']}' not recognized"
                    )
        except Exception as e:
            print(f"Error normalizing ore units for row {idx}: {e}")
            # Keep original values if there's an error

    return df


def normalize_grade_units(df):
    """
    Normalize grade units to percent. Creates two new columns:
    - normalized_grade_value: The grade value converted to percent
    - normalized_grade_unit: Set to '%' (percent)

    Args:
        df (pd.DataFrame): DataFrame containing grade values and units

    Returns:
        pd.DataFrame: DataFrame with added normalization columns
    """
    # Create copies of the original columns
    df["normalized_grade_value"] = df["grade_value"].copy()
    df["normalized_grade_unit"] = df["grade_unit_observed_name"].copy()

    # Define conversion factors to percent (%)
    conversion_factors = {
        "%": 1.0,  # Already in percent
        "percent": 1.0,  # Spelled out percent
        "g/t": 0.0001,  # Grams per tonne (1 g/t = 0.0001%)
        "g/mt": 0.0001,  # Grams per metric tonne
        "g/tonne": 0.0001,  # Grams per tonne (spelled out)
        "grams/tonne": 0.0001,  # Grams per tonne (fully spelled)
        "ppm": 0.0001,  # Parts per million (1 ppm = 0.0001%)
        "parts per million": 0.0001,  # Parts per million (spelled out)
        "oz/t": 0.00343,  # Troy ounces per short ton (1 oz/t ≈ 0.00343%)
        "oz/ton": 0.00343,  # Troy ounces per ton
        "opt": 0.00343,  # Ounces per ton abbreviation
        "kg/t": 0.1,  # Kilograms per tonne (1 kg/t = 0.1%)
        "ppb": 0.0000001,  # Parts per billion (1 ppb = 0.0000001%)
        "wt%": 1.0,  # Weight percent
        "wt.%": 1.0,  # Weight percent (alternative notation)
        "grams per tonne": 0.0001,  # Grams per tonne (spelled out)
        "gram per tonne": 0.0001,  # Grams per tonne (spelled out)
    }

    # Apply conversion
    for idx, row in df.iterrows():
        try:
            if pd.notna(row["grade_unit_observed_name"]) and pd.notna(
                row["grade_value"]
            ):
                unit = (
                    row["grade_unit_observed_name"].strip().lower()
                )  # Convert to lowercase for case-insensitive matching
                if unit in conversion_factors:
                    # Convert the value to percent
                    df.at[idx, "normalized_grade_value"] = (
                        float(row["grade_value"]) * conversion_factors[unit]
                    )
                    df.at[idx, "normalized_grade_unit"] = "%"
                else:
                    # If unit not recognized raise an error
                    raise ValueError(
                        f"Grade unit '{row['grade_unit_observed_name']}' not recognized"
                    )
        except Exception as e:
            print(f"Error normalizing grade units for row {idx}: {e}")
            # Keep original values if there's an error

    return df


master_inventory_df = normalize_ore_units(master_inventory_df)
master_inventory_df = normalize_grade_units(master_inventory_df)

# Assert there are no values in the "zone" column contain "total" to avoid double counting
assert not master_inventory_df["zone"].str.lower().str.contains("total").any()

In [26]:
category_to_resource_or_reserve = {
    "inferred": "resource",
    "measured": "resource",
    "indicated": "resource",
    "mineral resource": "resource",
    "measured+indicated": "resource",  # Exception
    "proven+probable": "reserve",  # Exception
    "proved": "reserve",
    "probable": "reserve",
    "proven": "reserve",
    "mineralresource": "resource",
}
# remove any whitespace from category (e.g. "proven + probable" -> "proven+probable")
master_inventory_df["category_observed_name"] = (
    master_inventory_df["category_observed_name"].str.replace(" ", "").str.lower()
)

master_inventory_df.insert(
    4,
    "resource_or_reserve",
    master_inventory_df["category_observed_name"].map(category_to_resource_or_reserve),
)
# Assert for all rows have "category_observed_name" it has a corresponding "resource_or_reserve" value
assert (
    not master_inventory_df[
        master_inventory_df["category_observed_name"].notna()
        & master_inventory_df["resource_or_reserve"].isna()
    ]
    .any()
    .any()
)


master_inventory_df["contained_metal"] = (
    master_inventory_df["normalized_ore_value"]
    * master_inventory_df["normalized_grade_value"]
    / 100
)

# Site-Level Ore and Contained Metal

In [27]:
selected_columns = [
    InferlinkEvalColumns.CDR_RECORD_ID.value,
    InferlinkEvalColumns.MAIN_COMMODITY.value,
    InferlinkEvalColumns.COMMODITY_OBSERVED_NAME.value,
    "resource_or_reserve",
    "normalized_ore_value",
    "contained_metal",
]


# Note: mineral site with no inventory data will have a NaN value in the "category_observed_name" column
# and thus a NaN value in the "resource_or_reserve" column. These rows will be dropped during the groupby operation.

# Group by cdr_record_id, resource_or_reserve and sum the normalized_ore_value
resource_n_reserve_total_tonnage = (
    master_inventory_df[selected_columns]
    .groupby(
        [
            InferlinkEvalColumns.CDR_RECORD_ID.value,
            InferlinkEvalColumns.MAIN_COMMODITY.value,
            InferlinkEvalColumns.COMMODITY_OBSERVED_NAME.value,
            "resource_or_reserve",
        ]
    )
    .agg(
        {
            "normalized_ore_value": "sum",
            "contained_metal": "sum",
        }
    )
    .reset_index()
)
resource_n_reserve_total_tonnage.head()

,cdr_record_id,main_commodity,commodity_observed_name,resource_or_reserve,normalized_ore_value,contained_metal
0,0200a1c6d2cfafeb485d815d95966961d4c119e8662b8b...,nickel,cobalt,resource,1171.486,0.126520
1,0200a1c6d2cfafeb485d815d95966961d4c119e8662b8b...,nickel,nickel,resource,1171.486,2.606034
2,0204fc707f5b1944308624520cd422c4f1cb478046f664...,tungsten,tungsten,reserve,1.465,0.004897
3,0204fc707f5b1944308624520cd422c4f1cb478046f664...,tungsten,tungsten,resource,4.086,0.010975
4,020548e1aca4c1ca222149e11a79f15bd7594c02eb1216...,earth_metals,dysprosium,resource,20.500,0.001301


In [28]:
# pivot the resource_n_reserve column to wide format
resource_n_reserve_total_tonnage = resource_n_reserve_total_tonnage.pivot(
    index=[
        InferlinkEvalColumns.CDR_RECORD_ID.value,
        InferlinkEvalColumns.MAIN_COMMODITY.value,
        InferlinkEvalColumns.COMMODITY_OBSERVED_NAME.value,
    ],
    columns="resource_or_reserve",
    values=["normalized_ore_value", "contained_metal"],
)
# expand the index to separate columns
resource_n_reserve_total_tonnage.columns = [
    f"{col[0]}_{col[1]}" for col in resource_n_reserve_total_tonnage.columns
]
resource_n_reserve_total_tonnage = resource_n_reserve_total_tonnage.reset_index()
resource_n_reserve_total_tonnage.head()

,cdr_record_id,main_commodity,commodity_observed_name,normalized_ore_value_reserve,normalized_ore_value_resource,contained_metal_reserve,contained_metal_resource
0,0200a1c6d2cfafeb485d815d95966961d4c119e8662b8b...,nickel,cobalt,NaN,1171.486,NaN,0.126520
1,0200a1c6d2cfafeb485d815d95966961d4c119e8662b8b...,nickel,nickel,NaN,1171.486,NaN,2.606034
2,0204fc707f5b1944308624520cd422c4f1cb478046f664...,tungsten,tungsten,1.465,4.086,0.004897,0.010975
3,020548e1aca4c1ca222149e11a79f15bd7594c02eb1216...,earth_metals,dysprosium,NaN,20.500,NaN,0.001301
4,020548e1aca4c1ca222149e11a79f15bd7594c02eb1216...,earth_metals,lanthanum,NaN,20.500,NaN,0.011928


In [29]:
# Left join master_metadata_df and master_inventory_df on cdr_record_id
master_df = pd.merge(
    resource_n_reserve_total_tonnage,
    master_metadata_df,
    on=InferlinkEvalColumns.CDR_RECORD_ID.value,
    how="left",
)

cols_to_drop = [
    "authors",
    "year",
    "month",
]
master_df = master_df.drop(columns=cols_to_drop)

cols_to_rename = {
    "normalized_ore_value_resource": InferlinkEvalColumns.TOTAL_MINERAL_RESOURCE_TONNAGE.value,
    "normalized_ore_value_reserve": InferlinkEvalColumns.TOTAL_MINERAL_RESERVE_TONNAGE.value,
    "contained_metal_resource": InferlinkEvalColumns.TOTAL_MINERAL_RESOURCE_CONTAINED_METAL.value,
    "contained_metal_reserve": InferlinkEvalColumns.TOTAL_MINERAL_RESERVE_CONTAINED_METAL.value,
    "country_observed_name": InferlinkEvalColumns.COUNTRY.value,
    "state_or_province_observed_name": InferlinkEvalColumns.STATE_OR_PROVINCE.value,
    "mining_name": InferlinkEvalColumns.MINERAL_SITE_NAME.value,
}
master_df = master_df.rename(columns=cols_to_rename)

# Add unique identifier at the first column
master_df.insert(0, InferlinkEvalColumns.ID.value, range(1, len(master_df) + 1))

# Map "mvt_zinc" to "zinc" for the "main_commodity" column
master_df[InferlinkEvalColumns.MAIN_COMMODITY.value] = master_df[
    InferlinkEvalColumns.MAIN_COMMODITY.value
].str.replace("mvt_", "")

master_df.to_csv(
    "paper/data/processed/ground_truth/inferlink_ground_truth.csv", index=False
)

## Val-Test Split

In [36]:
# Random sample roughly 50% of the cdr_record_id as the test set and the rest as the validation set. Set the seed to 42.
# Set the seed to 42
np.random.seed(42)

# Read the dataframe
df = pd.read_csv("paper/data/processed/ground_truth/inferlink_ground_truth.csv")

# Get unique cdr_record_ids
unique_cdr_ids = df[InferlinkEvalColumns.CDR_RECORD_ID.value].unique()

# Randomly sample 50% of unique cdr_record_ids for test set
test_cdr_ids = np.random.choice(
    unique_cdr_ids, size=len(unique_cdr_ids) // 2, replace=False
)

# Create test and validation sets
test_df = df[df[InferlinkEvalColumns.CDR_RECORD_ID.value].isin(test_cdr_ids)].copy()
val_df = df[~df[InferlinkEvalColumns.CDR_RECORD_ID.value].isin(test_cdr_ids)].copy()

# Sort by ID to maintain consistent ordering
test_df = test_df.sort_values(InferlinkEvalColumns.ID.value)
val_df = val_df.sort_values(InferlinkEvalColumns.ID.value)

# Save the splits
test_df.to_csv(
    "paper/data/processed/ground_truth/inferlink_ground_truth_test.csv", index=False
)
val_df.to_csv(
    "paper/data/processed/ground_truth/inferlink_ground_truth_val.csv", index=False
)

print(
    f"Test set size: {len(test_df)} | Unique CDR Report count: {test_df[InferlinkEvalColumns.CDR_RECORD_ID.value].nunique()}"
)
print(
    f"Validation set size: {len(val_df)} | Unique CDR Report count: {val_df[InferlinkEvalColumns.CDR_RECORD_ID.value].nunique()}"
)
print(
    f"Combined set size: {len(df)} | Unique CDR Report count: {df[InferlinkEvalColumns.CDR_RECORD_ID.value].nunique()}"
)

Test set size: 70 | Unique CDR Report count: 26
Validation set size: 79 | Unique CDR Report count: 27
Combined set size: 149 | Unique CDR Report count: 53


# Inventory-Level Ore and Grades


In [42]:
df = master_inventory_df[
    master_inventory_df[InferlinkEvalColumns.COMMODITY_OBSERVED_NAME.value].notna()
]
cols_to_drop = [
    "ore_unit_observed_name",
    "grade_unit_observed_name",
    "ore_value",
    "grade_value",
    "cutoff_grade_unit_observed_name",
    "cutoff_grade_value",
    "contained_metal",
    "zone",
]
df = df.drop(columns=cols_to_drop)
df.head()

,cdr_record_id,main_commodity,commodity_observed_name,category_observed_name,resource_or_reserve,normalized_ore_value,normalized_ore_unit,normalized_grade_value,normalized_grade_unit
1,02163e7674b5c838a0efd7d2e80116e36ba01cbdb1d892...,copper,copper,indicated,resource,2.827047,Mt,1.900000,%
2,02163e7674b5c838a0efd7d2e80116e36ba01cbdb1d892...,copper,zinc,indicated,resource,2.827047,Mt,0.320000,%
3,02163e7674b5c838a0efd7d2e80116e36ba01cbdb1d892...,copper,silver,indicated,resource,2.827047,Mt,0.000896,%
4,02163e7674b5c838a0efd7d2e80116e36ba01cbdb1d892...,copper,gold,indicated,resource,2.827047,Mt,0.000047,%
5,021ee3d3ec6928901d9427796372edaada0d332f15d030...,copper,copper,proven,reserve,93.475785,Mt,0.423000,%


# Parse PDF to Markdown